In [1]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest_asyncio
!pip -q install flash-attn --no-build-isolation



In [2]:
!nvidia-smi

Wed Dec 17 14:33:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from huggingface_hub import login

# You will be prompted to paste your HF token (make a token at: https://huggingface.co/settings/tokens)
login()


In [4]:
from fastapi import FastAPI, Query
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import re
import time
from typing import List, Dict, Any, Tuple, Optional

# =========================================================
# MODEL LOADING (QUALITY-FIRST, LARGER MODEL FOR H200)
# =========================================================

MODEL_NAME = "meta-llama/Meta-Llama-3-70B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model():
    assert torch.cuda.is_available(), "GPU required (H200 expected)"

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # Try best-quality first (BF16 on H200)
    try:
        print("Loading model (BF16 + FlashAttention2)...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.bfloat16,
            device_map={"": 0},
            attn_implementation="flash_attention_2",
        )
        model.eval()
        return model
    except Exception as e:
        print("BF16 load failed, falling back to 4-bit quantized load:", e)

    # Fallback: 4-bit (works if BF16 doesn't fit for some reason)
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print("Loading model (4-bit NF4)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
        attn_implementation="flash_attention_2",
    )
    model.eval()
    return model

model = load_model()
DEVICE = "cuda:0"
print(f"Using device: {DEVICE}")

# =========================================================
# LABELS
# =========================================================

LABELS = [
    "INVENTION","COMPONENT","SUBSYSTEM","MATERIAL","CHEMICAL","BIOMOLECULE","COMPOSITION",
    "PROCESS_STEP","METHOD","PARAMETER","MEASUREMENT","CONDITION","FUNCTION","SIGNAL",
    "CONTROL","SOFTWARE","HARDWARE","FIGURE_REF","CLAIM_ELEMENT","PRIOR_ART","UNCLASSIFIED_ENTITY",
]

# =========================================================
# FASTAPI APP
# =========================================================

app = FastAPI()

class PredictRequest(BaseModel):
    data: list  # [{"text": "..."}]

# =========================================================
# HELPERS (ROBUST JSON EXTRACTION + SPAN RECOVERY)
# =========================================================

_JSON_ARRAY_RE = re.compile(r"\[\s*\{.*?\}\s*\]", flags=re.DOTALL)

def _extract_json_array(s: str) -> List[Dict[str, Any]]:
    m = _JSON_ARRAY_RE.search(s)
    if not m:
        l = s.find("[")
        r = s.rfind("]")
        if l == -1 or r == -1 or r <= l:
            return []
        blob = s[l:r+1]
    else:
        blob = m.group(0)

    try:
        parsed = json.loads(blob)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

def _find_span(text: str, entity_text: str, start_from: int = 0) -> Tuple[int, int]:
    idx = text.find(entity_text, start_from)
    if idx == -1:
        return (-1, -1)
    return (idx, idx + len(entity_text))

def _normalize_for_match(s: str) -> str:
    s = s.strip().strip('"').strip("'")
    s = re.sub(r"\s+", " ", s)
    return s

def _try_recover_span(text: str, ent_text: str) -> Tuple[int, int]:
    ent = _normalize_for_match(ent_text)
    if not ent:
        return (-1, -1)

    start, end = _find_span(text, ent)
    if start != -1:
        return (start, end)

    t_low = text.lower()
    e_low = ent.lower()
    idx = t_low.find(e_low)
    if idx != -1:
        return (idx, idx + len(ent))

    return (-1, -1)

# =========================================================
# QUALITY-FIRST PROMPT (CHAT TEMPLATE + EXAMPLE + HIGH RECALL)
# =========================================================

def _build_prompt(text: str) -> str:
    labels_str = ", ".join(LABELS)

    system = (
        "You are a high-recall patent NER engine. "
        "Your job is to find as many plausible entities as possible."
    )

    user = f"""
Extract span entities from the input text.
Use ONLY these labels:
{labels_str}

Rules:
- Return ONLY a valid JSON array (no markdown, no explanations).
- Each item MUST be an object with keys: "text" and "label".
- "text" MUST be copied VERBATIM from the input text (exact substring).
- Aim for high recall: include any plausible entity spans.
- Overlapping spans are allowed.

Example:

Input text:
"A temperature sensor is connected to a control unit via a wireless interface."

Output:
[
  {{"text":"temperature sensor","label":"COMPONENT"}},
  {{"text":"control unit","label":"COMPONENT"}},
  {{"text":"wireless interface","label":"SUBSYSTEM"}}
]

Now process this input text:

\"\"\"{text}\"\"\"

Return JSON array now:
""".strip()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# =========================================================
# CORE NER LOGIC (NO OVERLAP FILTERING)
# =========================================================

def extract_spans(text: str) -> List[Dict[str, Any]]:
    prompt = _build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    gen_kwargs = dict(
        max_new_tokens=400,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    t0 = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    t1 = time.time()

    prompt_len = inputs["input_ids"].shape[-1]
    gen_ids = output[0][prompt_len:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)

    dt = max(t1 - t0, 1e-6)
    print(f"[NER] {gen_ids.shape[-1]} tokens in {dt:.2f}s", flush=True)

    items = _extract_json_array(decoded)

    spans: List[Dict[str, Any]] = []
    for it in items:
        if not isinstance(it, dict):
            continue

        ent_text = str(it.get("text", "")).strip()
        label = str(it.get("label", "")).strip()

        if not ent_text or label not in LABELS:
            continue

        start, end = _find_span(text, ent_text)
        if start == -1:
            start, end = _try_recover_span(text, ent_text)
            if start == -1:
                continue

        spans.append({"start": start, "end": end, "text": text[start:end], "labels": [label]})

    return spans

# =========================================================
# ROUTES
# =========================================================

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE, "model": MODEL_NAME}

@app.api_route("/setup", methods=["GET", "POST"])
def setup():
    return {
        "from_name": "label",
        "to_name": "text",
        "type": "labels",
        "labels": LABELS,
    }

@app.api_route("/predict", methods=["GET", "POST"])
def predict(
    request: Optional[PredictRequest] = None,
    text: Optional[str] = Query(default=None),
):
    if request is not None and request.data:
        data = request.data
    elif text is not None:
        data = [{"text": text}]
    else:
        return []

    print(f"🔵 /predict received: n_items={len(data)}", flush=True)

    results = []
    for item in data:
        t = item.get("text", "")
        if not t:
            results.append({"result": [], "score": 0.0})
            continue

        spans = extract_spans(t)

        results.append(
            {
                "result": [
                    {
                        "from_name": "label",
                        "to_name": "text",
                        "type": "labels",
                        "value": span,
                    }
                    for span in spans
                ],
                "score": 1.0,
            }
        )

    return results


Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/note

Loading model (BF16 + FlashAttention2)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Using device: cuda:0


In [5]:
import uvicorn, threading, nest_asyncio, time, requests

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# start server in background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(5)  # give it a few seconds to start

# quick test: local health
print(requests.get("http://127.0.0.1:8000/health").json())


INFO:     Started server process [26996]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:45478 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'device': 'cuda:0', 'model': 'meta-llama/Meta-Llama-3-8B-Instruct'}


In [6]:
from google.colab import output

public_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
public_url


'https://8000-gpu-a100-s-3ilu1aphsdoti-f.us-central1-1.prod.colab.dev'

In [7]:
from pyngrok import ngrok
from dotenv import load_dotenv
load_dotenv()
import os
# kill old tunnels in this session
ngrok.kill()

# your auth token
ngrok.set_auth_token("2PgsprcdKolcczw6ru6HXbLcYfC_7cUSXnTdho7wqZyHYotoF")

# A) random domain
# public_url = ngrok.connect(addr="127.0.0.1:8000")

# B) your reserved free domain
public_url = ngrok.connect(
    addr="127.0.0.1:8000",
    domain="empiristic-mariyah-unprophetically.ngrok-free.dev"
)

print("Public URL:", public_url)


Public URL: NgrokTunnel: "https://empiristic-mariyah-unprophetically.ngrok-free.dev" -> "http://127.0.0.1:8000"
